# Task 3 conditional 3D U-Net
Thin Colab orchestration for the isolated package. This copy is configured for a fresh training run through full validation inference. Final packaging still requires a live rules review, and nothing uploads a submission.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

BASELINE_MILESTONE_COMPLETE = True
ACKNOWLEDGE_OVER_BUDGET = False
STAGE_RUNTIME = True
STAGE_RUN_BACKUPS = False
RUN_PACKAGE_TESTS = True
RUN_DATA_AUDIT = True
PREPARE_PROSPECTIVE_CACHE = True
RUN_TINY_SMOKE_UPDATES = True
RUN_A100_PROBE = True
RUN_RETROSPECTIVE_PRETRAINING = True
RUN_THREE_ABLATIONS = True
SELECT_WINNER = True
RUN_FINAL_RETRAINING = True
RUN_VALIDATION_SMOKE = True
RUN_FULL_VALIDATION = True
REFRESH_OFFICIAL_SOURCES = True
LIVE_RULES_REVIEWED = False
BUILD_TASK3_ZIP = False
EXPORT_SELECTED_ARTIFACTS = True
EXPORT_ARCHIVE = False

DRIVE_MOUNT = Path('/content/drive')
DRIVE_CODE_ZIP = DRIVE_MOUNT / 'MyDrive/mrixfields/task3_3d_unet_source.zip'
DRIVE_DATA_ROOT = DRIVE_MOUNT / 'MyDrive/mrixfields2026'
DRIVE_BACKUP_ROOT = DRIVE_MOUNT / 'MyDrive/mrixfields/task3_3d_unet_backups'
PROJECT_ROOT = Path('/content/mri_competition')
RUNTIME_DATA = Path('/content/task3_3d_unet_data')
CACHE_DIR = Path('/content/task3_3d_unet_cache')
RUN_ROOT = Path('/content/task3_3d_unet_runs')
OFFICIAL_REPO = Path('/content/MRIxFields2026')
RUNTIME_REPORT = RUN_ROOT / 'a100_probe.json'
EXPECTED_LIVE_COMMIT = ''
EXPECTED_SUBMISSION_README_SHA256 = ''

def run(*parts):
    subprocess.check_call([str(part) for part in parts], cwd=PROJECT_ROOT)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def assert_runtime_only():
    if DRIVE_MOUNT.is_mount():
        raise RuntimeError('Drive must be unmounted before processing')


In [ ]:
if STAGE_RUNTIME:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
    if not DRIVE_CODE_ZIP.is_file():
        raise FileNotFoundError(DRIVE_CODE_ZIP)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_CODE_ZIP) as archive:
        archive.extractall(PROJECT_ROOT)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
        str(PROJECT_ROOT / 'task3_3d_unet/requirements-colab.txt')])
    run(sys.executable, '-m', 'task3_3d_unet.data', 'stage',
        '--data-dir', DRIVE_DATA_ROOT, '--runtime-dir', RUNTIME_DATA)
    if STAGE_RUN_BACKUPS and DRIVE_BACKUP_ROOT.is_dir():
        for relative in ('pretrain', 'ablations', 'final'):
            source_root = DRIVE_BACKUP_ROOT / relative
            if source_root.is_dir():
                target_root = RUN_ROOT / relative
                shutil.copytree(source_root, target_root, dirs_exist_ok=True)
                for source in source_root.rglob('*'):
                    if source.is_file() and sha256(source) != sha256(target_root / source.relative_to(source_root)):
                        raise RuntimeError(f'Restored backup hash mismatch: {source}')
        shared_source = DRIVE_BACKUP_ROOT / '_shared'
        if shared_source.is_dir():
            for source in shared_source.iterdir():
                if source.is_file():
                    target = RUN_ROOT / source.name
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, target)
                    if sha256(source) != sha256(target):
                        raise RuntimeError(f'Restored backup hash mismatch: {source}')
    drive.flush_and_unmount()
    print('Runtime staging complete. Drive is unmounted.')


In [ ]:
if REFRESH_OFFICIAL_SOURCES:
    assert_runtime_only()
    if OFFICIAL_REPO.exists():
        raise RuntimeError('Use a fresh path for the organizer checkout')
    subprocess.check_call(['git', 'clone', '--depth', '1',
        'https://github.com/MRIxFields/MRIxFields2026.git', str(OFFICIAL_REPO)])
    commit = subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    readme_hash = sha256(OFFICIAL_REPO / 'Submission/README.md')
    print({'official_commit': commit, 'submission_readme_sha256': readme_hash})
    print('Review the competition page and Submission README. If they disagree, stop and leave LIVE_RULES_REVIEWED false.')


In [ ]:
if RUN_PACKAGE_TESTS:
    assert_runtime_only()
    run(sys.executable, '-m', 'unittest', 'discover', '-v', 'task3_3d_unet/tests')

if RUN_DATA_AUDIT:
    assert_runtime_only()
    run(sys.executable, '-m', 'task3_3d_unet.data', 'audit',
        '--data-dir', RUNTIME_DATA, '--report', RUN_ROOT / 'data_audit.json')

if PREPARE_PROSPECTIVE_CACHE:
    assert_runtime_only()
    run(sys.executable, '-m', 'task3_3d_unet.data', 'prepare-cache',
        '--data-dir', RUNTIME_DATA, '--cache-dir', CACHE_DIR,
        '--report', RUN_ROOT / 'cache_report.json')


In [ ]:
if RUN_TINY_SMOKE_UPDATES:
    assert_runtime_only()
    assert BASELINE_MILESTONE_COMPLETE
    run(sys.executable, '-m', 'task3_3d_unet.train', 'pretrain',
        '--retrospective-manifest', RUNTIME_DATA / 'retrospective_subset.json',
        '--output-dir', RUN_ROOT / 'tiny_pretrain', '--total-updates', '2',
        '--patch-size', '16', '--block-size', '4', '--batch-size', '1',
        '--gradient-accumulation', '1', '--checkpoint-every', '1',
        '--max-runtime-minutes', '0', '--device', 'cuda', '--amp', 'bf16',
        '--tiny', '--baseline-milestone-complete')
    run(sys.executable, '-m', 'task3_3d_unet.train', 'finetune',
        '--cache-dir', CACHE_DIR, '--official-repo', OFFICIAL_REPO,
        '--arm', 'paired_masked', '--output-dir', RUN_ROOT / 'tiny_finetune',
        '--total-updates', '2', '--patch-size', '16', '--batch-size', '1',
        '--gradient-accumulation', '1', '--eval-every', '2', '--skip-lpips',
        '--checkpoint-every', '1', '--max-runtime-minutes', '0',
        '--device', 'cuda', '--amp', 'bf16', '--tiny', '--baseline-milestone-complete')

if RUN_A100_PROBE:
    assert_runtime_only()
    run(sys.executable, '-m', 'task3_3d_unet.probe',
        '--cache-dir', CACHE_DIR, '--official-repo', OFFICIAL_REPO,
        '--report', RUNTIME_REPORT, '--updates', '100')


In [ ]:
def backup_stopped_chunk(run_dir):
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
    destination = DRIVE_BACKUP_ROOT / run_dir.relative_to(RUN_ROOT)
    destination.mkdir(parents=True, exist_ok=True)
    provenance_path = run_dir / 'provenance.json'
    complete = provenance_path.is_file() and json.loads(provenance_path.read_text())['status'] == 'complete'
    checkpoint = 'checkpoints/best.pt' if complete and (run_dir / 'checkpoints/best.pt').is_file() else 'checkpoints/last.pt'
    stale = destination / ('checkpoints/last.pt' if checkpoint.endswith('best.pt') else 'checkpoints/best.pt')
    stale.unlink(missing_ok=True)
    for relative in (checkpoint, 'provenance.json', 'summary.json', 'metrics.jsonl'):
        source = run_dir / relative
        if source.is_file():
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
            if sha256(source) != sha256(target):
                raise RuntimeError(f'Backup hash mismatch: {source}')
    shared_destination = DRIVE_BACKUP_ROOT / '_shared'
    shared_destination.mkdir(parents=True, exist_ok=True)
    for source in (RUNTIME_REPORT, RUN_ROOT / 'winner.json', RUN_ROOT / 'data_audit.json', RUN_ROOT / 'cache_report.json'):
        if source.is_file():
            target = shared_destination / source.name
            shutil.copy2(source, target)
            if sha256(source) != sha256(target):
                raise RuntimeError(f'Backup hash mismatch: {source}')
    drive.flush_and_unmount()

def run_chunked(command, run_dir, maximum_chunks=20):
    existing = run_dir / 'provenance.json'
    if existing.is_file() and json.loads(existing.read_text())['status'] == 'complete':
        return
    for _ in range(maximum_chunks):
        assert_runtime_only()
        resume = run_dir / 'checkpoints/last.pt'
        current = list(command) + (['--resume', str(resume)] if resume.is_file() else [])
        run(*current)
        backup_stopped_chunk(run_dir)
        provenance = json.loads((run_dir / 'provenance.json').read_text())
        if provenance['status'] == 'complete':
            return
    raise RuntimeError('Chunk limit reached before training completed')

def paid_flags():
    flags = ['--baseline-milestone-complete', '--runtime-report', str(RUNTIME_REPORT)]
    if ACKNOWLEDGE_OVER_BUDGET:
        flags.append('--acknowledge-over-budget')
    return flags


In [ ]:
if RUN_RETROSPECTIVE_PRETRAINING:
    assert_runtime_only()
    assert BASELINE_MILESTONE_COMPLETE
    command = [sys.executable, '-m', 'task3_3d_unet.train', 'pretrain',
        '--retrospective-manifest', str(RUNTIME_DATA / 'retrospective_subset.json'),
        '--output-dir', str(RUN_ROOT / 'pretrain'), '--total-updates', '20000',
        '--patch-size', '96', '--block-size', '12', '--mask-fraction', '0.5',
        '--patches-per-volume', '64', '--batch-size', '2',
        '--gradient-accumulation', '2', '--amp', 'bf16',
        '--max-runtime-minutes', '45'] + paid_flags()
    run_chunked(command, RUN_ROOT / 'pretrain')


In [ ]:
if RUN_THREE_ABLATIONS:
    assert_runtime_only()
    assert BASELINE_MILESTONE_COMPLETE
    for arm in ('paired_global', 'paired_masked', 'pretrained_masked'):
        run_dir = RUN_ROOT / 'ablations' / arm
        command = [sys.executable, '-m', 'task3_3d_unet.train', 'finetune',
            '--cache-dir', str(CACHE_DIR), '--official-repo', str(OFFICIAL_REPO),
            '--arm', arm, '--output-dir', str(run_dir), '--total-updates', '10000',
            '--patch-size', '96', '--batch-size', '2', '--gradient-accumulation', '2',
            '--eval-every', '1000', '--amp', 'bf16', '--max-runtime-minutes', '45']
        if arm == 'pretrained_masked':
            command += ['--pretrained', str(RUN_ROOT / 'pretrain/checkpoints/best.pt')]
        run_chunked(command + paid_flags(), run_dir)

if SELECT_WINNER:
    assert_runtime_only()
    run(sys.executable, '-m', 'task3_3d_unet.compare',
        '--ablation-root', RUN_ROOT / 'ablations', '--report', RUN_ROOT / 'winner.json')


In [ ]:
if RUN_FINAL_RETRAINING:
    assert_runtime_only()
    assert BASELINE_MILESTONE_COMPLETE
    selection = json.loads((RUN_ROOT / 'winner.json').read_text())
    arm = selection['winner']['arm']
    run_dir = RUN_ROOT / 'final'
    command = [sys.executable, '-m', 'task3_3d_unet.train', 'finetune',
        '--cache-dir', str(CACHE_DIR), '--official-repo', str(OFFICIAL_REPO),
        '--arm', arm, '--all-subjects', '--output-dir', str(run_dir),
        '--selection-report', str(RUN_ROOT / 'winner.json'),
        '--total-updates', '10000', '--patch-size', '96', '--batch-size', '2',
        '--gradient-accumulation', '2', '--amp', 'bf16', '--max-runtime-minutes', '45']
    if arm == 'pretrained_masked':
        command += ['--pretrained', str(RUN_ROOT / 'pretrain/checkpoints/best.pt')]
    run_chunked(command + paid_flags(), run_dir)


In [ ]:
if RUN_VALIDATION_SMOKE:
    assert_runtime_only()
    inference_batch = str(json.loads(RUNTIME_REPORT.read_text())['inference_batch_size'])
    run(sys.executable, '-m', 'task3_3d_unet.infer',
        '--checkpoint', RUN_ROOT / 'final/checkpoints/best.pt',
        '--input-root', RUNTIME_DATA, '--output-dir', RUN_ROOT / 'smoke_inference',
        '--window', '128', '--overlap', '0.25', '--batch-size', inference_batch,
        '--amp', 'bf16', '--smoke')

if RUN_FULL_VALIDATION:
    assert_runtime_only()
    inference_batch = str(json.loads(RUNTIME_REPORT.read_text())['inference_batch_size'])
    run(sys.executable, '-m', 'task3_3d_unet.infer',
        '--checkpoint', RUN_ROOT / 'final/checkpoints/best.pt',
        '--input-root', RUNTIME_DATA, '--output-dir', RUN_ROOT / 'full_inference',
        '--window', '128', '--overlap', '0.25', '--batch-size', inference_batch,
        '--smoke-manifest', RUN_ROOT / 'smoke_inference/task3_3d_unet_inference_manifest.json',
        '--amp', 'bf16', '--resume')


In [ ]:
if BUILD_TASK3_ZIP:
    assert_runtime_only()
    assert LIVE_RULES_REVIEWED
    assert EXPECTED_LIVE_COMMIT and EXPECTED_SUBMISSION_README_SHA256
    subprocess.check_call(['git', '-C', str(OFFICIAL_REPO), 'fetch', 'origin', 'main'])
    remote_commit = subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'origin/main'], text=True).strip()
    if remote_commit != EXPECTED_LIVE_COMMIT:
        raise RuntimeError(f'Official source changed after review: {remote_commit}')
    run(sys.executable, '-m', 'task3_3d_unet.submission',
        '--official-repo', OFFICIAL_REPO, '--predictions-dir', RUN_ROOT / 'full_inference',
        '--data-dir', RUNTIME_DATA, '--output-dir', RUN_ROOT / 'submission',
        '--archive', RUN_ROOT / 'task3.zip', '--report', RUN_ROOT / 'submission_report.json',
        '--runtime-report', RUNTIME_REPORT, '--selection-report', RUN_ROOT / 'winner.json',
        '--expected-official-commit', EXPECTED_LIVE_COMMIT,
        '--expected-submission-readme-sha256', EXPECTED_SUBMISSION_README_SHA256,
        '--live-rules-reviewed')


In [ ]:
if EXPORT_SELECTED_ARTIFACTS:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
    export_root = DRIVE_BACKUP_ROOT / 'selected'
    export_root.mkdir(parents=True, exist_ok=True)
    selected = [RUN_ROOT / 'winner.json', RUN_ROOT / 'a100_probe.json',
                RUN_ROOT / 'final/checkpoints/best.pt', RUN_ROOT / 'submission_report.json']
    if EXPORT_ARCHIVE:
        selected.append(RUN_ROOT / 'task3.zip')
    for source in selected:
        if source.is_file():
            target = export_root / source.name
            shutil.copy2(source, target)
            if sha256(source) != sha256(target):
                raise RuntimeError(f'Export hash mismatch: {source}')
    drive.flush_and_unmount()
    print('Selected artifacts exported. No submission upload was performed.')
